# ⚡ dspyer Playground

Welcome to the official **dspyer** interactive playground! Here you will learn how to transpile graph-based agent topologies into declarative, optimizable DSPy programs in seconds.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theramkm/dspyer/blob/main/notebooks/dspyer_playground.ipynb)

### Why dspyer?
- **Stateful Imperative Graphs**: Build agent workflows as structured graphs of nodes and edges.
- **Declarative Modules**: Automatically transpile graphs into standard `dspy.Module` programs.
- **Auto-Optimization**: Directly optimize your compiled prompt pipelines with DSPy teleprompters (e.g. `BootstrapFewShot`).

## 🛠️ 1. Setup & Installation

First, install `dspyer` and its required dependencies. In Colab, we'll install from PyPI (or GitHub).

In [ ]:
# Install dspyer and DSPy
%pip install dspyer dspy-ai pydantic json-repair httpx

## 🚀 2. 10-Second Quickstart (Mock / No-Key Mode)

Let's define a simple extraction & classification workflow. We will run it with a zero-config local `MockLM` to ensure everything executes instantly, without requiring external LLM API keys.

In [ ]:
import dspy
from pydantic import BaseModel, Field

from dspy_transpiler.compiler import AgentTranspiler
from dspy_transpiler.graph import Graph, StatefulNode


# 1. Define input & output schemas
class ExtractionInput(BaseModel):
    raw_text: str = Field(description="The source text to analyze")


class ExtractionOutput(BaseModel):
    user_name: str = Field(description="Name of the user extracted from text")
    rough_query: str = Field(description="User query context")


class ClassificationInput(BaseModel):
    rough_query: str = Field(description="Extracted rough query")


class ClassificationOutput(BaseModel):
    classified_intent: str = Field(description="Category intent: Support, Sales, Info")


# 2. Build graph nodes
node_extraction = StatefulNode(
    name="EntityExtractor",
    input_model=ExtractionInput,
    output_model=ExtractionOutput,
    instructions="Extract the user's name and their query context from raw text.",
)

node_classification = StatefulNode(
    name="IntentClassifier",
    input_model=ClassificationInput,
    output_model=ClassificationOutput,
    instructions="Classify the intent of the rough query into: Support, Sales, or Info.",
)

# 3. Build graph topology
graph = Graph()
graph.add_node(node_extraction)
graph.add_node(node_classification)
graph.set_entry_point("EntityExtractor")
graph.add_edge("EntityExtractor", "IntentClassifier")

# 4. Transpile graph into a declarative DSPy module
program = AgentTranspiler.compile(graph)


# 5. Create a Mock LM to intercept LLM calls
class QuickstartMockLM(dspy.LM):
    def __init__(self):
        super().__init__(model="mock-quickstart-model")

    def forward(self, prompt=None, messages=None, **kwargs):
        prompt_str = str(prompt or messages)
        if "user_name" in prompt_str:
            content = '{"user_name": "Alice", "rough_query": "buy a subscription"}'
        elif "classified_intent" in prompt_str:
            content = '{"classified_intent": "Sales"}'
        else:
            content = "{}"

        class MockChoiceMessage:
            def __init__(self, content_str):
                self.content = content_str
                self.role = "assistant"
                self.reasoning_content = None

        class MockChoice:
            def __init__(self, content_str):
                self.message = MockChoiceMessage(content_str)
                self.finish_reason = "stop"
                self.index = 0

        class MockResult:
            def __init__(self, content_str):
                self.choices = [MockChoice(content_str)]
                self.model = "mock-quickstart-model"
                self.usage = {"completion_tokens": 0, "prompt_tokens": 0, "total_tokens": 0}

        return MockResult(content)


# 6. Configure DSPy to use the Mock LM and run
dspy.configure(lm=QuickstartMockLM())
result = program(raw_text="Hello, my name is Alice. I would like to buy a subscription.")
print("Transpiled program output:")
print(result)

## 🛡️ 3. RAG Self-Correction / Verification Loop

In production, workflows need guardrails. Here is an Answer Synthesizer node that validates whether it generated citations. If it fails validation, `dspyer` runs a refinement loop sending the validation error back to the model for self-correction.

In [ ]:
from pydantic import field_validator


class UserQuery(BaseModel):
    query: str = Field(description="The query asked by the user")


class RetrievedDocuments(BaseModel):
    query: str = Field(description="The user's query")
    context: str = Field(description="The retrieved chunks and background context")


class RAGResponse(BaseModel):
    answer: str = Field(description="The synthesized answer referencing the context")
    citations: list[str] = Field(description="List of source document names cited (e.g. ['doc_1'])")

    @field_validator("citations")
    @classmethod
    def must_have_citations(cls, val):
        if not val or len(val) == 0:
            raise ValueError("Synthesized answer must contain at least one valid source citation.")
        return val


# Set up retriever and synthesizer nodes
node_retriever = StatefulNode(
    name="DocRetriever",
    input_model=UserQuery,
    output_model=RetrievedDocuments,
    instructions="Retrieve relevant documents and text chunks for the query.",
)

node_synthesizer = StatefulNode(
    name="AnswerSynthesizer",
    input_model=RetrievedDocuments,
    output_model=RAGResponse,
    instructions="Synthesize a cited answer based on the retrieved context.",
)

graph = Graph()
graph.add_node(node_retriever)
graph.add_node(node_synthesizer)
graph.set_entry_point("DocRetriever")
graph.add_edge("DocRetriever", "AnswerSynthesizer")

program = AgentTranspiler.compile(graph)


# Create Mock LM that fails on the first pass (no citations) and succeeds on the correction pass
class RAGMockLM(dspy.LM):
    def __init__(self):
        super().__init__(model="mock-rag-model")

    def forward(self, prompt=None, messages=None, **kwargs):
        prompt_str = str(prompt or messages)
        if "context" in prompt_str and "citations" not in prompt_str:
            content = '{"query": "What is dspyer?", "context": "[doc_1] dspyer is released under the Apache-2.0 License."}'
        elif "citations" in prompt_str:
            if "failed_output" in prompt_str or "feedback" in prompt_str:
                content = '{"answer": "dspyer is licensed under the Apache-2.0 License as of 2026 [doc_1].", "citations": ["doc_1"]}'
            else:
                content = '{"answer": "dspyer is licensed under the Apache-2.0 License.", "citations": []}'
        else:
            content = "{}"

        class MockChoiceMessage:
            def __init__(self, content_str):
                self.content = content_str
                self.role = "assistant"
                self.reasoning_content = None

        class MockChoice:
            def __init__(self, content_str):
                self.message = MockChoiceMessage(content_str)
                self.finish_reason = "stop"
                self.index = 0

        class MockResult:
            def __init__(self, content_str):
                self.choices = [MockChoice(content_str)]
                self.model = "mock-rag-model"
                self.usage = {"completion_tokens": 0, "prompt_tokens": 0, "total_tokens": 0}

        return MockResult(content)


dspy.configure(lm=RAGMockLM())
result = program(query="What is the license of dspyer?")
print("RAG Verifier Output:")
print(f"Answer: {result.answer}")
print(f"Citations: {result.citations}")
print(f"Self-correction steps: {result['_metadata']['refinement_steps_taken']}")

## 🎯 4. Prompt Optimization via DSPy BootstrapFewShot

Since transpiled graphs are native `dspy.Module` classes, you can use DSPy's optimizers to tune prompt instructions/few-shot examples based on training data. Let's optimize a French translator agent node.

In [ ]:
from dspy.teleprompt import BootstrapFewShot


class TranslationInput(BaseModel):
    english_text: str = Field(description="The source text in English")


class TranslationOutput(BaseModel):
    french_text: str = Field(description="The translated text in French")


translation_node = StatefulNode(
    name="Translator",
    input_model=TranslationInput,
    output_model=TranslationOutput,
    instructions="Translate the provided English text into French.",
)

graph = Graph()
graph.add_node(translation_node)
graph.set_entry_point("Translator")
program = AgentTranspiler.compile(graph)


class OptimizerMockLM(dspy.LM):
    def __init__(self):
        super().__init__(model="mock-optimizer-model")

    def forward(self, prompt=None, messages=None, **kwargs):
        prompt_str = str(prompt or messages)
        if "Hello, how are you?" in prompt_str:
            content = '{"french_text": "Bonjour, comment ça va ?"}'
        elif "Thank you very much" in prompt_str:
            content = '{"french_text": "Merci beaucoup"}'
        elif "Good morning" in prompt_str:
            content = '{"french_text": "Bonjour"}'
        else:
            content = '{"french_text": "Salut"}'

        class MockChoiceMessage:
            def __init__(self, content_str):
                self.content = content_str
                self.role = "assistant"
                self.reasoning_content = None

        class MockChoice:
            def __init__(self, content_str):
                self.message = MockChoiceMessage(content_str)
                self.finish_reason = "stop"
                self.index = 0

        class MockResult:
            def __init__(self, content_str):
                self.choices = [MockChoice(content_str)]
                self.model = "mock-optimizer-model"
                self.usage = {"completion_tokens": 0, "prompt_tokens": 0, "total_tokens": 0}

        return MockResult(content)


dspy.configure(lm=OptimizerMockLM())

# Define few-shot training dataset
trainset = [
    dspy.Example(
        english_text="Hello, how are you?", french_text="Bonjour, comment ça va ?"
    ).with_inputs("english_text"),
    dspy.Example(english_text="Thank you very much", french_text="Merci beaucoup").with_inputs(
        "english_text"
    ),
]


# Simple validation metric
def exact_match_metric(example, pred, trace=None):
    return example.french_text.strip().lower() == pred.french_text.strip().lower()


# Run BootstrapFewShot optimizer
optimizer = BootstrapFewShot(metric=exact_match_metric, max_bootstrapped_demos=2)
optimized_program = optimizer.compile(program, trainset=trainset)

print("\nOptimized Program Predictors:")
for name, predictor in optimized_program.named_predictors():
    demos = predictor.demos if hasattr(predictor, "demos") else []
    print(f"Predictor '{name}' now has {len(demos)} bootstrapped few-shot examples.")

## 📊 5. Tracing & Telemetry Integration

To trace and visualize execution in production, standard OpenTelemetry configurations can pipe traces directly to tools like **Arize Phoenix**, **Langfuse**, or **Jaeger**.

Run the following to start Arize Phoenix locally (or via Colab) to visualize execution traces:

In [ ]:
# Start Phoenix trace server and register OpenTelemetry exporter
try:
    import phoenix as px
    from openinference.instrumentation.dspy import DSPyInstrumentor
    from opentelemetry import trace
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import SimpleSpanProcessor

    # Launch phoenix local UI
    px.launch()

    # Setup OpenTelemetry provider pointing to local Phoenix port
    otlp_exporter = OTLPSpanExporter(endpoint="http://localhost:6006/v1/traces")
    tracer_provider = TracerProvider()
    tracer_provider.add_span_processor(SimpleSpanProcessor(otlp_exporter))
    trace.set_tracer_provider(tracer_provider)

    # Instrument DSPy calls
    DSPyInstrumentor().instrument()
    print("Tracing active! Go to http://localhost:6006/ to visualize your traces.")
except ImportError:
    print(
        "Install extra packages to run telemetry locally: pip install arize-phoenix openinference-instrumentation-dspy opentelemetry-exporter-otlp"
    )